# Use Case 9: Backup, Restore & Database Administration

**The Concept:** 
Production databases need operational tools: backups, restore points, compression tuning, storage statistics, garbage collection, and WAL management. SochDB provides all of these as simple API calls.

**The Architecture:** 
SochDB uses an append-only MVCC storage engine with a Write-Ahead Log (WAL). This notebook covers the full admin toolkit: creating/verifying backups, restoring from snapshots, tuning compression, inspecting storage stats, and performing maintenance.

---

### Step 0: Install Packages

In [ ]:
!pip install sochdb

import os
import json
import time

### Step 1: Initialize Database & Populate Data
Create a database with some data to work with.

In [ ]:
from sochdb import Database

db = Database.open("./admin_demo_db")

# Populate with some data
for i in range(100):
    db.put(f"record:{i:04d}".encode(), json.dumps({"id": i, "value": f"data_{i}", "timestamp": int(time.time())}).encode())

db.execute("CREATE TABLE users (id INT, name TEXT, email TEXT)")
db.execute("INSERT INTO users VALUES (1, 'Alice', 'alice@example.com')")
db.execute("INSERT INTO users VALUES (2, 'Bob', 'bob@example.com')")
db.execute("INSERT INTO users VALUES (3, 'Carol', 'carol@example.com')")

print(f"Database populated with 100 KV records and 3 SQL rows.")
print(f"Database path: {db.db_path()}")

### Step 2: Storage Statistics
Inspect current storage usage and database health.

In [ ]:
# Basic stats
stats = db.stats()
print("Basic Database Stats:")
print(json.dumps(stats, indent=2, default=str))

print("\n" + "="*50)

# Full detailed stats
full_stats = db.stats_full()
print("\nFull Database Stats:")
print(json.dumps(full_stats, indent=2, default=str))

### Step 3: Compression Configuration
Tune compression for your workload: `none`, `lz4` (fast), `zstd_fast` (balanced), `zstd_max` (smallest).

In [ ]:
# Check current compression
current = db.get_compression()
print(f"Current compression: {current}")

# Switch to LZ4 for fast read/write workloads
db.set_compression("lz4")
print(f"Compression set to: {db.get_compression()}")

# Switch to zstd_max for maximum compression (archival workloads)
db.set_compression("zstd_max")
print(f"Compression set to: {db.get_compression()}")

# Reset to balanced
db.set_compression("zstd_fast")
print(f"Compression set to: {db.get_compression()}")

### Step 4: Checkpointing
Flush the in-memory memtable to disk. Critical before backups.

In [ ]:
# Standard checkpoint — flush active memtable
db.checkpoint()
print("Standard checkpoint completed.")

# Full checkpoint — flush all memtables and compact
db.checkpoint_full()
print("Full checkpoint completed.")

# Force durable sync to disk (fsync)
db.fsync()
print("Fsync completed — all data durable on disk.")

### Step 5: Create a Backup
Create a consistent point-in-time backup of the entire database.

In [ ]:
BACKUP_DIR = "./admin_demo_backups"
os.makedirs(BACKUP_DIR, exist_ok=True)

backup_path = os.path.join(BACKUP_DIR, "backup_001")
db.backup_create(backup_path)
print(f"Backup created at: {backup_path}")

### Step 6: Verify Backup Integrity
Validate that a backup is complete and uncorrupted.

In [ ]:
# Verify the backup
is_valid = Database.backup_verify(backup_path)
print(f"Backup integrity check: {'PASS' if is_valid else 'FAIL'}")

# List all backups in the directory
backups = Database.backup_list(BACKUP_DIR)
print(f"\nAvailable backups:")
for b in backups:
    print(f"  {b}")

### Step 7: Modify Data, Then Restore from Backup
Delete some data, then restore the database to the backup snapshot.

In [ ]:
# Delete 50 records
for i in range(50):
    db.delete(f"record:{i:04d}".encode())

# Verify data was deleted
remaining = db.scan_prefix(b"record:")
print(f"Records after deletion: {len(remaining)} (deleted 50)")

# Restore from backup
db.backup_restore(backup_path)
print("Database restored from backup!")

# Verify restoration
restored = db.scan_prefix(b"record:")
print(f"Records after restore: {len(restored)} (all 100 recovered)")

### Step 8: Garbage Collection
Clean up old MVCC versions that are no longer needed.

In [ ]:
# Run garbage collection to reclaim space from old MVCC versions
gc_count = db.gc()
print(f"Garbage collection completed. Reclaimed {gc_count} old version(s).")

### Step 9: WAL Management
Truncate the Write-Ahead Log after a successful checkpoint.

In [ ]:
# First checkpoint to ensure all data is persisted
db.checkpoint_full()

# Then truncate the WAL to reclaim disk space
db.truncate_wal()
print("WAL truncated after checkpoint.")

### Step 10: Table Index Policies
Control how tables are indexed for query performance.

In [ ]:
# Set index policy for the users table
db.set_table_index_policy("users", "eager")
print(f"Users table index policy: {db.get_table_index_policy('users')}")

# Verify the table still works
result = db.execute("SELECT * FROM users")
print(f"\nUsers table ({len(result.rows)} rows):")
for row in result.rows:
    print(f"  {row}")

### Step 11: TOON & JSON Serialization Formats
SochDB includes efficient wire formats for token-optimized serialization.

In [ ]:
# Standard JSON serialization
data = {"name": "Alice", "scores": [95, 87, 92], "metadata": {"role": "engineer"}}

json_bytes = Database.to_json(data)
print(f"JSON format ({len(json_bytes)} bytes): {json_bytes}")

restored = Database.from_json(json_bytes)
print(f"Restored: {restored}")

print()

# TOON format (Token-Optimized Object Notation) — more compact
toon_bytes = Database.to_toon(data)
print(f"TOON format ({len(toon_bytes)} bytes): {toon_bytes}")

restored_toon = Database.from_toon(toon_bytes)
print(f"Restored: {restored_toon}")

print(f"\nSpace savings: {len(json_bytes) - len(toon_bytes)} bytes smaller with TOON")

### Cleanup

In [ ]:
# Graceful shutdown
db.shutdown()
print("Database shut down gracefully.")